In [3]:
from langchain_community.document_loaders import TextLoader
from pathlib import Path

path_to_dir = "../data/01_raw/"
# print(path_to_dir.exists())      # 해당경로에 파일이 실제로 존재하는지

In [7]:
import glob
import os
import pymupdf4llm
from langchain_core.documents import Document

path_to_dir = "../data/01_raw/"
documents = []

# 폴더 내의 모든 pdf 파일을 탐색
pdf_files = glob.glob(os.path.join(path_to_dir, "*.pdf"))

for pdf_path in pdf_files:
    # PDF를 마크다운 페이지 단위로 로드
    md_pages = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)
    
    # enumerate를 사용하여 안전하게 페이지 번호(idx) 확보
    for idx, page in enumerate(md_pages):
        page_meta = page.get("metadata", {})
        
        # 'page_nr' 또는 'page' 키 확인, 둘 다 없으면 idx(0부터 시작) 사용
        page_num = page_meta.get("page_nr", page_meta.get("page", idx + 1)) - 1
        total_pages = page_meta.get("page_count", len(md_pages))
        
        doc = Document(
            page_content=page["text"],
            metadata={
                "source": pdf_path,
                "page": page_num,
                "total_pages": total_pages
            }
        )
        documents.append(doc)

print(f"로드된 총 페이지 수: {len(documents)}")

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

로드된 총 페이지 수: 1529


In [8]:
print(f"로드된 문서 수: {len(documents)}")
print(f"문서 길이: {len(documents[0].page_content)} 글자")
print(f"메타데이터: {documents[0].metadata}")
print(f"\n미리보기 (처음 300자):")
print(documents[0].page_content[:300])

로드된 문서 수: 1529
문서 길이: 6 글자
메타데이터: {'source': '../data/01_raw/2022_주택·상가건물 임대차분쟁조정 사례집.pdf', 'page': 0, 'total_pages': 152}

미리보기 (처음 300자):









In [9]:
import re

# 1. 텍스트 정제 함수 정의
def clean_page_content(text: str) -> str:
    # 양끝 공백 제거 및 줄바꿈(\n)을 일반 공백으로 변경
    text = text.strip().replace("\n", " ")
    # 연속된 공백(두 칸 이상)을 하나의 공백으로 축소
    text = re.sub(r'\s+', ' ', text)
    return text

# 2. 전처리 적용 및 빈 페이지 필터링
cleaned_documents = []
for doc in documents:
    cleaned_text = clean_page_content(doc.page_content)
    
    # 전처리 후 실제 텍스트 내용이 남아있는 페이지만 저장
    if len(cleaned_text.strip()) > 0:
        doc.page_content = cleaned_text
        cleaned_documents.append(doc)

print(f"전처리 전 페이지 수: {len(documents)}장")
print(f"전처리 후 페이지 수: {len(cleaned_documents)}장")

전처리 전 페이지 수: 1529장
전처리 후 페이지 수: 980장


In [10]:
# 10번째 페이지(인덱스 9)가 존재한다면 정제된 내용 미리보기
if len(cleaned_documents) > 9:
    print(f"문서 출처: {cleaned_documents[9].metadata.get('source')}")
    print(f"페이지 번호: {cleaned_documents[9].metadata.get('page')}")
    print(f"정제된 문서 길이: {len(cleaned_documents[9].page_content)} 글자")
    print("\n--- 전처리 후 미리보기 (처음 300자) ---")
    print(cleaned_documents[9].page_content[:300])
else:
    print("텍스트가 포함된 페이지가 10장 미만입니다.")

문서 출처: ../data/01_raw/2024_주택·상가건물 임대차분쟁조정 사례집.pdf
페이지 번호: 7
정제된 문서 길이: 538 글자

--- 전처리 후 미리보기 (처음 300자) ---
###### 7 손해배상 > Q 임대인의 실거주 갱신거절이 허위로 밝혀진 경우 손해배상청구가 가능한가요? > Q 계약갱신을 거절한 후 언제까지 실거주 하여야 손해배상 책임이 없는 것인지, 이혼을 한 경우 정당한 갱신거절 사유에 해당하는지 궁금합니다. > Q 임대인이 실거주 갱신거절 후 매매를 할 경우 계약갱신요구가 가능한지 및 손해배상 청구 > Q 누수로 인하여 임대차계약이 해지된 경우 임대인에게 손해배상을 청구할 수 있나요? ###### 8 중개사보수 > Q 가계약이 파기되었는데 중개보수를 지급해야 하나요? > Q 계약갱신요구에 


In [ ]:
md_file_path = os.path.join(processed_dir, "cleaned_preview_check.md")

with open(md_file_path, "w", encoding="utf-8") as f:
    f.write("# 전처리 완료 데이터 확인용 마크다운 파일\n\n")
    f.write(f"- **총 로드된 유효 페이지 수:** {len(cleaned_documents)}장\n\n")
    f.write("---\n\n")
    
    # 처음 10개 페이지의 정제 데이터 샘플링 기록
    for i in range(min(10, len(cleaned_documents))):
        doc = cleaned_documents[i]
        source_name = os.path.basename(doc.metadata.get('source', 'unknown'))
        page_num = doc.metadata.get('page', i)
        
        f.write(f"## 📄 샘플 {i+1}. 파일명: {source_name} (PDF 내 {page_num}페이지)\n")
        f.write(f"- **글자 수:** {len(doc.page_content)}자\n")
        f.write(f"- **내용:**\n> {doc.page_content}\n\n")
        f.write("---\n\n")

print(f"🎉 검증용 마크다운 파일 생성 완료! 경로: {md_file_path}")
print("이제 해당 경로의 파일을 열어서 줄바꿈이나 공백이 이쁘게 합쳐졌는지 확인해보세요.")

In [11]:
import os
import json
from collections import defaultdict

# 1. 타겟 경로를 PyMuPDF4LLM 하위 폴더로 정확하게 변경
processed_dir = "../data/03_processed/PyMuPDF4LLM/"
os.makedirs(processed_dir, exist_ok=True)

# 2. 전체 전처리 데이터(cleaned_documents)를 원본 파일명별로 그룹핑
grouped_data = defaultdict(list)
for doc in cleaned_documents:
    source_path = doc.metadata.get('source')
    # 전체 경로에서 실제 파일명만 추출
    file_name = os.path.basename(source_path) 
    
    # 해당 파일 그룹에 페이지 내용과 메타데이터 적재
    grouped_data[file_name].append({
        "page_content": doc.page_content,
        "metadata": doc.metadata
    })

print("--- 파일별 03_processed/PyMuPDF4LLM 저장 시작 ---")

# 3. 그룹핑된 데이터를 파일별로 각각 JSON 저장
for file_name, pages in grouped_data.items():
    # 확장자를 .pdf에서 _processed.json으로 변경
    output_file_name = file_name.replace(".pdf", "_processed.json")
    output_path = os.path.join(processed_dir, output_file_name)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=4)
        
    print(f"✅ 저장 완료: {output_file_name} (총 {len(pages)}개 페이지)")

print("\n🎉 모든 파일이 개별 JSON으로 data/03_processed/PyMuPDF4LLM/ 폴더에 성공적으로 저장되었습니다!")

--- 파일별 03_processed/PyMuPDF4LLM 저장 시작 ---
✅ 저장 완료: 20231006_개정_주택임대차 표준계약서_특약사항 포함(개정후)_processed.json (총 5개 페이지)
✅ 저장 완료: 2024_주택·상가건물 임대차분쟁조정 사례집_processed.json (총 108개 페이지)
✅ 저장 완료: 2024_주택임대차 상담사례집(최종_배포)_processed.json (총 108개 페이지)
✅ 저장 완료: 2021_[한국부동산원] 주택·상가건물 임대차분쟁조정사례집_processed.json (총 180개 페이지)
✅ 저장 완료: 2025_주택 및 상가건물 임대차 상담사례집_국토교통부_한국부동산원_(최종_배포)_processed.json (총 121개 페이지)
✅ 저장 완료: 주택임대차전월세계약시 주요 확인사항_20260313_processed.json (총 5개 페이지)
✅ 저장 완료: 전월세종합지원센터_챗봇서비스_전세사기 상담 지원_서울특별시_processed.json (총 7개 페이지)
✅ 저장 완료: 2022_개정_주택임대차 표준계약서(개정전)_processed.json (총 5개 페이지)
✅ 저장 완료: 2021_주택·상가건물 임대차분쟁조정 사례집_processed.json (총 180개 페이지)
✅ 저장 완료: 주택임대차보호법 가이드북_20200731_개정판_processed.json (총 131개 페이지)
✅ 저장 완료: 소송등 법적절차 안내문(서울시 전월세종합지원센터)_processed.json (총 9개 페이지)
✅ 저장 완료: 2025_주택·상가건물 임대차분쟁조정 사례집_processed.json (총 121개 페이지)

🎉 모든 파일이 개별 JSON으로 data/03_processed/PyMuPDF4LLM/ 폴더에 성공적으로 저장되었습니다!
